In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, BitsAndBytesConfig
import soundfile as sf

# Configuration for 8-bit quantization
quantization_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # Threshold for outlier detection
    llm_int8_skip_modules=["lm_heads"],  # Skip language modeling heads
)

# Configuration for 4-bit quantization (more aggressive)
quantization_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat4 quantization
    bnb_4bit_use_double_quant=True,  # Nested quantization for better compression
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for inference
)

# Load model with quantization
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Choose one configuration
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-large-v1",
    quantization_config=quantization_config_8bit,  # or quantization_config_4bit
    device_map="auto",  # Automatically distribute across devices
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-large-v1")

In [ ]:
# Test inference
prompt = "Hey, how are you doing today?"
description = "A female speaker delivers a slightly expressive and animated speech with a moderate speed and pitch."

input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# Generate audio
with torch.inference_mode():
    generation = model.generate(
        input_ids=input_ids, 
        prompt_input_ids=prompt_input_ids,
        do_sample=True,
        temperature=1.0
    )

audio_arr = generation.cpu().numpy().squeeze()
sf.write("quantized_parler_tts_out.wav", audio_arr, model.config.sampling_rate)

print(f"Model size reduced. Audio generated successfully!")


### Sensitivity-Aware Mixed Precision (SAMP) Implementation

In [ ]:
from typing import Set


def apply_sensitivity_aware_quantization(model, sensitive_modules: Set[str]):
    """
    Apply mixed precision quantization based on module sensitivity.
    
    Args:
        model: The ParlerTTS model to quantize
        sensitive_modules: Set of module name patterns to keep in FP16
    
    Returns:
        Quantized model with mixed precision
    """
    from bitsandbytes.nn import Linear8bitLt
    import torch.nn as nn
    
    # Track quantization statistics
    total_params = 0
    quantized_params = 0
    skipped_layers = []
    quantized_layers = []
    
    def is_sensitive_module(name: str) -> bool:
        """Check if module name matches any sensitive pattern"""
        for pattern in sensitive_modules:
            if pattern in name:
                return True
        return False
    
    # Iterate through all named modules
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            total_params += module.weight.numel()
            
            # Check if this is a sensitive module that should stay in FP16
            if is_sensitive_module(name):
                skipped_layers.append(name)
                print(f"  [SKIP] Keeping {name} in FP16 (sensitive)")
                continue
            
            # Quantize this layer to INT8
            try:
                # Get parent module and attribute name
                *parent_names, attr_name = name.rsplit('.', 1) if '.' in name else ('', name)
                parent = model
                if parent_names:
                    for pname in parent_names[0].split('.'):
                        parent = getattr(parent, pname)
                
                # Create 8-bit linear layer
                quantized_layer = Linear8bitLt(
                    module.in_features,
                    module.out_features,
                    bias=module.bias is not None,
                    has_fp16_weights=False,  # Use INT8 weights
                    threshold=6.0  # Outlier threshold for mixed precision decomposition
                )
                
                # Copy weights and bias
                quantized_layer.weight = nn.Parameter(module.weight.data.clone())
                if module.bias is not None:
                    quantized_layer.bias = nn.Parameter(module.bias.data.clone())
                
                # Replace the module
                setattr(parent, attr_name, quantized_layer)
                quantized_params += module.weight.numel()
                quantized_layers.append(name)
                print(f"  [QUANT] Quantized {name} to INT8")
                
            except Exception as e:
                print(f"  [ERROR] Failed to quantize {name}: {e}")
                skipped_layers.append(name)
    
    # Print summary
    print("\n" + "="*70)
    print("QUANTIZATION SUMMARY")
    print("="*70)
    print(f"Total linear layers: {len(quantized_layers) + len(skipped_layers)}")
    print(f"Quantized to INT8: {len(quantized_layers)}")
    print(f"Kept in FP16: {len(skipped_layers)}")
    print(f"Parameters quantized: {quantized_params:,} / {total_params:,} "
          f"({100*quantized_params/total_params:.1f}%)")
    print("="*70 + "\n")
    
    return model

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

# Define sensitive modules to keep in FP16
# Based on SAMP: decoder attention layers are most sensitive
SENSITIVE_MODULES = {
    # Decoder self-attention (critical for autoregressive generation)
    "decoder.block",
    "self_attn",
    
    # Decoder cross-attention (critical for text-to-audio alignment)
    "cross_attn",
    "encoder_attn",
    
    # Final projection layers (critical for output quality)
    "lm_head",
    "final_proj",
}

In [ ]:
# Load model in FP16 (baseline precision)
print("Loading ParlerTTS model...")
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-large-v1",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
print(f"✓ Model loaded in FP16\n")

In [ ]:
# Apply SAMP quantization
model = apply_sensitivity_aware_quantization(model, SENSITIVE_MODULES)
model = model.to(device)
model.eval()

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-large-v1")
print("Tokenizer loaded\n")

In [ ]:
# Test prompts
prompt = "Hey, how are you doing today?"
description = "A female speaker delivers a slightly expressive and animated speech with a moderate speed and pitch."

print(f"Prompt: '{prompt}'")
print(f"Description: '{description}'\n")

# Tokenize inputs
input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# Generate audio with timing
import time
print("Generating audio...")
start_time = time.time()

with torch.inference_mode():
    generation = model.generate(
        input_ids=input_ids, 
        prompt_input_ids=prompt_input_ids,
        do_sample=True,
        temperature=1.0,
        max_new_tokens=1000  # Limit generation length
    )

end_time = time.time()
generation_time = end_time - start_time

# Save audio
audio_arr = generation.cpu().numpy().squeeze()
output_path = "samp_quantized_parler_tts_out.wav"
sf.write(output_path, audio_arr, model.config.sampling_rate)

# Calculate metrics
audio_duration = len(audio_arr) / model.config.sampling_rate
rtf = generation_time / audio_duration

print(f"Audio generated and saved to '{output_path}'")
print(f"Generation time: {generation_time:.2f} seconds")
print(f"Audio duration: {audio_duration:.2f} seconds")
print(f"Real-Time Factor (RTF): {rtf:.4f}")
